# Dimension branches


In [0]:
#Imports
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [0]:

#full snapshot every run
branches_df = spark.read.table("neo_bank.silver.branches")
dim_table = DeltaTable.forName(spark, "neo_bank.gold.dim_branches")

(
    dim_table.alias("target")
    .merge(
        branches_df.alias("source"),
        condition="target.branch_code = source.branch_code"
    )
    .whenMatchedUpdate(set={
        "branch_name": "source.branch_name",
        "city": "source.city",
        "state": "source.state",
        "region": "source.region"
    })
    .whenNotMatchedInsert(values={
        "branch_code": "source.branch_code",
        "branch_name": "source.branch_name",
        "city": "source.city",
        "state": "source.state",
        "region": "source.region"
    })
    .execute()
)